In [ ]:
! pip install librosa
! pip install pandas
! pip install -U scikit-learn
! pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
!pip -q install pandas numpy scikit-learn torch transformers accelerate sentence-transformers
!pip -q install huggingface_hub ollama python-dotenv imbalanced-learn

In [ ]:
! unzip datos_competicion_SpM.zip

Archive:  datos_competicion_SpM.zip
   creating: test/
  inflating: test/audios_test.zip    
  inflating: test/corpus_ironia_iberlef2026_test.csv  
   creating: train/
  inflating: train/audios_train.zip  
  inflating: train/corpus_ironia_iberlef2026_train.csv  


In [ ]:
! chmod u+w train
! chmod u+w test

! mkdir train/audios
! unzip train/audios_train.zip -d train/audios

! mkdir test/audios
! unzip test/audios_test.zip -d test/audios

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: train/audios/audios_flac/2067.flac  
  inflating: train/audios/audios_flac/2068.flac  
  inflating: train/audios/audios_flac/2069.flac  
  inflating: train/audios/audios_flac/207.flac  
  inflating: train/audios/audios_flac/2070.flac  
  inflating: train/audios/audios_flac/2071.flac  
  inflating: train/audios/audios_flac/2074.flac  
  inflating: train/audios/audios_flac/2075.flac  
  inflating: train/audios/audios_flac/2076.flac  
  inflating: train/audios/audios_flac/2078.flac  
  inflating: train/audios/audios_flac/2079.flac  
  inflating: train/audios/audios_flac/208.flac  
  inflating: train/audios/audios_flac/2080.flac  
  inflating: train/audios/audios_flac/2081.flac  
  inflating: train/audios/audios_flac/2082.flac  
  inflating: train/audios/audios_flac/2083.flac  
  inflating: train/audios/audios_flac/2084.flac  
  inflating: train/audios/audios_flac/2085.flac  
  inflating: train/audios/audios_flac/208

In [ ]:
!sudo apt-get install zstd
# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el servidor de Ollama en segundo plano
import subprocess
import threading
import time

def run_ollama():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama)
thread.start()
time.sleep(5) # Dar tiempo a que el servidor inicie
print("Servidor de Ollama iniciado.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (9,972 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currentl

In [ ]:
# === Opción A: Ollama local ===
# Descarga modelos a tu instancia local de Ollama (puerto 11434).
# El prompt está diseñado para funcionar con cualquiera de ellos:
!ollama pull llama3.1:8b
#!ollama pull mistral:7b
!ollama pull gemma2:9b
#!ollama pull qwen2.5:7b
#!ollama pull qwen3:8b          # modelo con razonamiento (<think>...</think>)
#!ollama pull llama3.2:3b       # modelo pequeño rápido
#!ollama pull deepseek-r1:8b    # modelo con razonamiento
!ollama list

# === Opción B: Ollama Cloud / Ollama Web (sin descarga local) ===
# 1) Crea una API key en https://ollama.com/settings/keys
# 2) Expórtala como variable de entorno:
#    import os; os.environ["OLLAMA_API_KEY"] = "tu_clave_aqui"
# 3) Configura el pipeline para usar el endpoint cloud:
#    config["ollama_base_url"] = "https://ollama.com"
#    config["llm_model"]       = "gpt-oss:120b"   # u otro modelo cloud
# 4) Ejecuta normalmente: pipeline.run()
# No necesitas hacer 'ollama pull' para modelos cloud.




NAME           ID              SIZE      MODIFIED               
gemma2:9b      ff02c3702f32    5.4 GB    Less than a second ago    
llama3.1:8b    46e0c10c039e    4.9 GB    58 seconds ago            


In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
# =============================================================================
# 0. System setup (uncomment for Colab)
# =============================================================================
!pip -q install pandas numpy scikit-learn torch torchvision transformers accelerate
!pip -q install sentence-transformers pillow requests huggingface_hub ollama python-dotenv


In [ ]:
# -*- coding: utf-8 -*-
"""
SpeechMATICS Advanced Pipeline
==============================
Irony detection in spoken Spanish using embeddings, LLMs, and hybrid approaches.

Supports:
  - Sentence embeddings (HuggingFace sentence-transformers) + classifier
  - LLM zero/few-shot classification:
      * Ollama local (http://localhost:11434, sin auth)
      * Ollama Cloud / Ollama Web (https://ollama.com, requiere API key)
      * HuggingFace Inference API
  - MFCC audio features for multimodal fusion
  - Ensemble of multiple approaches

Prompt diseñado para funcionar con CUALQUIER modelo de Ollama:
  - modelos pequeños (llama3.2:3b, gemma2:2b, qwen2.5:3b)
  - modelos medianos (llama3.1:8b, gemma2:9b, mistral:7b)
  - modelos grandes (llama3.1:70b, gpt-oss:120b, qwen3:32b)
  - modelos con razonamiento / thinking (qwen3, deepseek-r1) — el parser
    descarta el bloque <think>...</think> antes de buscar la etiqueta

Usage:
  python speechmatics_pipeline.py --config config.yaml
  python speechmatics_pipeline.py --approach embedding --model bge-m3
  python speechmatics_pipeline.py --approach llm --provider ollama --llm-model llama3.1:8b
  # Ollama Cloud:
  export OLLAMA_API_KEY=tu_clave
  python speechmatics_pipeline.py --llm-provider ollama \
      --ollama-base-url https://ollama.com --llm-model gpt-oss:120b
"""

import os
import sys
import json
import argparse
import logging
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ============================================================================
# CONFIGURATION
# ============================================================================

DEFAULT_CONFIG = {
    # Paths
    "train_csv": "train/corpus_ironia_iberlef2026_train.csv",
    "test_csv": "test/corpus_ironia_iberlef2026_test.csv",
    "train_audio_dir": "train/audios/audios_flac/",
    "test_audio_dir": "test/audios/audios_flac/",
    "output_file": "submission_file.csv",

    # Task 1 (text-only) approach: "tfidf", "embedding", "llm", "ensemble"
    "task1_approach": "ensemble",

    # Task 2 (multimodal) approach: "tfidf_mfcc", "embedding_mfcc", "llm", "ensemble"
    "task2_approach": "ensemble",

    # Embedding settings
    "embedding_model": "intfloat/multilingual-e5-large-instruct", #"sentence-transformers/paraphrase-multilingual-mpnet-base-v2" (0.59),#"intfloat/multilingual-e5-large"(0.65) "google/embeddinggemma-300M" (0.64) "BAAI/bge-m3"(0.63) "nomic-ai/nomic-embed-text-v1"(0.62), #"hiiamsid/sentence_similarity_spanish_es",#"intfloat/multilingual-e5-large",#"sentence-transformers/LaBSE", #"nomic-ai/nomic-embed-text-v1.5", #"BAAI/bge-m3",  # Good multilingual model
    "embedding_device": "cuda",          # "cpu", "cuda", "mps"
    "embedding_batch_size": 32,
    "embedding_normalize": True,

    # LLM settings
    "llm_provider": "ollama",           # "ollama" or "huggingface"
    "llm_model": "qwen3.5:397b-cloud", #"deepseek-v4-flash:cloud" (), #"gemma4:31b-cloud (0.64101)",         # Model name for the provider
    # ollama_base_url:
    #   - Local:        "http://localhost:11434"  (sin auth)
    #   - Ollama Cloud: "https://ollama.com"      (requiere ollama_api_key)
    "ollama_base_url": "https://ollama.com",
    "ollama_api_key": "",             # Set via env OLLAMA_API_KEY. Solo necesario para Ollama Cloud
    "hf_api_token": None,               # Set via env HF_API_TOKEN
    "llm_temperature": 0.0,
    "llm_max_tokens": 128,              # Subido para dejar margen a modelos con <think>...</think>
    "llm_num_few_shot": 3,              # Number of few-shot examples per class
    "llm_request_timeout": 180,         # Ollama Cloud puede tardar más en la primera llamada (cold start)

    # Classifier settings (for embedding/tfidf approaches)
    "classifier": "logreg",                # "svm", "logreg", "mlp", "rf"
    "scaler": "standard",               # "minmax", "standard", "none"
    "grid_search": True,
    "cv_folds": 5,

    # Audio feature settings
    "audio_features": ["mfcc"],         # "mfcc", "chroma", "spectral_contrast", "tonnetz"
    "n_mfcc": 20,
    "audio_sr": 22050,

    # Ensemble settings
    "ensemble_methods": ["tfidf", "embedding","llm"],  # Methods to combine
    "ensemble_strategy": "soft_voting",           # "soft_voting", "hard_voting", "stacking"
}


def load_config(config_path: Optional[str] = None) -> dict:
    """Load configuration from YAML file or use defaults."""
    config = DEFAULT_CONFIG.copy()
    if config_path and os.path.exists(config_path):
        try:
            import yaml
            with open(config_path) as f:
                user_config = yaml.safe_load(f)
            if user_config:
                config.update(user_config)
            logger.info(f"Loaded config from {config_path}")
        except ImportError:
            logger.warning("PyYAML not installed; using JSON fallback")
            with open(config_path) as f:
                config.update(json.load(f))
    return config


# ============================================================================
# TEXT FEATURE EXTRACTORS
# ============================================================================

class TfidfFeatureExtractor:
    """Classic TF-IDF text features."""

    def __init__(self, max_features=10_000, analyzer="word", ngram_range=(1, 2)):
        self.vectorizer = TfidfVectorizer(
            analyzer=analyzer,
            max_features=max_features,
            lowercase=False,
            ngram_range=ngram_range,
            sublinear_tf=True,
        )
        self.fitted = False

    def fit_transform(self, texts):
        X = self.vectorizer.fit_transform(texts).toarray()
        self.fitted = True
        return X

    def transform(self, texts):
        return self.vectorizer.transform(texts).toarray()


class EmbeddingFeatureExtractor:
    """
    Sentence embeddings via sentence-transformers.
    Supports any HuggingFace model compatible with sentence-transformers.

    Recommended models for Spanish:
      - BAAI/bge-m3                (multilingual, 1024d, excellent)
      - intfloat/multilingual-e5-large  (multilingual, 1024d)
      - hiiamsid/sentence_similarity_spanish_es  (Spanish-specific, 768d)
      - sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2  (384d, fast)
    """

    def __init__(self, model_name="BAAI/bge-m3", device="cuda", batch_size=32,
                 normalize=True):
        self.model_name = model_name
        self.device = device
        self.batch_size = batch_size
        self.normalize = normalize
        self._model = None

    def _load_model(self):
        if self._model is None:
            from sentence_transformers import SentenceTransformer
            logger.info(f"Loading embedding model: {self.model_name}")
            self._model = SentenceTransformer(self.model_name, device=self.device,trust_remote_code=True)
            logger.info(f"Model loaded. Embedding dimension: {self._model.get_sentence_embedding_dimension()}")

    def _encode(self, texts):
        self._load_model()
        # For E5 models, prepend query/passage prefix
        if "e5" in self.model_name.lower():
            texts = [f"query: {t}" for t in texts]
        embeddings = self._model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=self.normalize,
        )
        return embeddings

    def fit_transform(self, texts):
        return self._encode(texts)

    def transform(self, texts):
        return self._encode(texts)


# ============================================================================
# AUDIO FEATURE EXTRACTORS
# ============================================================================

class AudioFeatureExtractor:
    """Extract acoustic features from audio files."""

    def __init__(self, features=None, n_mfcc=20, sr=22050):
        self.features = features or ["mfcc"]
        self.n_mfcc = n_mfcc
        self.sr = sr

    def extract_from_file(self, path: str) -> np.ndarray:
        import librosa
        data, sr = librosa.load(path, sr=self.sr)
        return self._extract(data, sr)

    def _extract(self, data, sr) -> np.ndarray:
        import librosa
        parts = []

        if "mfcc" in self.features:
            mfcc = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=self.n_mfcc)
            # Use mean and std for richer representation
            parts.append(np.mean(mfcc.T, axis=0))
            parts.append(np.std(mfcc.T, axis=0))

        if "chroma" in self.features:
            chroma = librosa.feature.chroma_stft(y=data, sr=sr)
            parts.append(np.mean(chroma.T, axis=0))
            parts.append(np.std(chroma.T, axis=0))

        if "spectral_contrast" in self.features:
            sc = librosa.feature.spectral_contrast(y=data, sr=sr)
            parts.append(np.mean(sc.T, axis=0))
            parts.append(np.std(sc.T, axis=0))

        if "tonnetz" in self.features:
            tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(data), sr=sr)
            parts.append(np.mean(tonnetz.T, axis=0))
            parts.append(np.std(tonnetz.T, axis=0))

        return np.hstack(parts)

    def extract_batch(self, ids, audio_dir, labels=None, label2id=None):
        X = []
        Y = [] if labels is not None else None
        failed = []

        for i, sample_id in enumerate(tqdm(ids, desc="Extracting audio features")):
            path = os.path.join(audio_dir, f"{sample_id}.flac")
            if not os.path.exists(path):
                # Try other extensions
                for ext in [".wav", ".mp3", ".ogg"]:
                    alt = os.path.join(audio_dir, f"{sample_id}{ext}")
                    if os.path.exists(alt):
                        path = alt
                        break

            try:
                feat = self.extract_from_file(path)
                X.append(feat)
                if labels is not None and label2id is not None:
                    Y.append(label2id[labels.iloc[i]])
            except Exception as e:
                logger.warning(f"Failed to process {path}: {e}")
                failed.append(sample_id)
                # Zero-fill on failure
                X.append(np.zeros(self._expected_dim()))
                if labels is not None and label2id is not None:
                    Y.append(label2id[labels.iloc[i]])

        if failed:
            logger.warning(f"{len(failed)} audio files failed to process")

        return np.array(X), np.array(Y) if Y is not None else None

    def _expected_dim(self):
        dim = 0
        if "mfcc" in self.features:
            dim += self.n_mfcc * 2  # mean + std
        if "chroma" in self.features:
            dim += 12 * 2
        if "spectral_contrast" in self.features:
            dim += 7 * 2
        if "tonnetz" in self.features:
            dim += 6 * 2
        return dim


# ============================================================================
# LLM CLASSIFIERS
# ============================================================================

class LLMClassifier:
    """
    Zero-shot or few-shot irony classification using LLMs.

    Soporta:
      - Ollama local         -> http://localhost:11434  (sin auth)
      - Ollama Cloud / Web   -> https://ollama.com      (Authorization: Bearer <API_KEY>)
      - HuggingFace Inference API

    Diseñado para funcionar con CUALQUIER modelo de Ollama:
      - modelos pequeños (llama3.2:3b, gemma2:2b, qwen2.5:3b...)
      - modelos medianos (llama3.1:8b, gemma2:9b, mistral:7b, qwen2.5:7b...)
      - modelos grandes (llama3.1:70b, gpt-oss:120b, qwen3:32b...)
      - modelos con razonamiento / thinking (qwen3, deepseek-r1, gpt-oss):
        el parser descarta cualquier bloque <think>...</think> antes de
        buscar la etiqueta final.

    El prompt usa un contrato estricto: la respuesta debe ir dentro de
    <respuesta>...</respuesta>. Esto permite que incluso modelos verbosos
    o "charlatanes" emitan la etiqueta de forma extraíble.
    """

    # System prompt: claro, breve, modelo-agnóstico, en español.
    # Evitamos jerga lingüística que confunda a modelos pequeños y dejamos
    # explícito el formato de salida.
    SYSTEM_PROMPT = (
        "Eres un clasificador de ironía en español hablado. "
        "Dado un fragmento, debes decidir si contiene ironía retórica "
        "(el sentido pretendido contradice el significado literal de las palabras) "
        "o no.\n\n"
        "REGLAS ESTRICTAS:\n"
        "1. Tu respuesta DEBE terminar con la etiqueta dentro de las etiquetas "
        "<respuesta> y </respuesta>.\n"
        "2. El contenido dentro de <respuesta> SOLO puede ser una de estas dos "
        "cadenas exactas:\n"
        "     ironia\n"
        "     no_ironia\n"
        "3. No expliques tu decisión. No añadas comentarios. No uses comillas. "
        "No uses tildes en la etiqueta.\n"
        "4. Si tienes dudas, responde no_ironia."
    )

    # Plantilla canónica usada tanto en few-shot como en la consulta final.
    # Mantenemos el mismo formato exacto en ejemplos y query para no
    # confundir a modelos pequeños.
    _USER_TEMPLATE = (
        'Fragmento: "{text}"\n'
        'Responde con <respuesta>ironia</respuesta> o <respuesta>no_ironia</respuesta>.'
    )
    _ASSISTANT_TEMPLATE = "<respuesta>{label}</respuesta>"

    def __init__(self, provider="ollama", model="llama3.1:8b",
                 base_url="http://localhost:11434",
                 api_key=None, hf_token=None,
                 temperature=0.0, max_tokens=128, num_few_shot=3,
                 request_timeout=180):
        self.provider = provider
        self.model = model
        # Normaliza base_url: sin barra final
        self.base_url = (base_url or "http://localhost:11434").rstrip("/")
        # API key para Ollama Cloud (env var OLLAMA_API_KEY como fallback)
        self.api_key = api_key or os.environ.get("OLLAMA_API_KEY")
        self.hf_token = hf_token or os.environ.get("HF_API_TOKEN")
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.num_few_shot = num_few_shot
        self.request_timeout = request_timeout
        self.few_shot_examples = []

        # Aviso si el usuario apunta a Ollama Cloud sin API key
        if self.provider == "ollama" and self._is_cloud_url() and not self.api_key:
            logger.warning(
                "Estás usando un endpoint de Ollama Cloud (%s) pero no se ha "
                "proporcionado una API key. Configura ollama_api_key o exporta "
                "OLLAMA_API_KEY. Las llamadas fallarán con 401.",
                self.base_url,
            )

    def _is_cloud_url(self) -> bool:
        """Detecta si la URL base apunta a Ollama Cloud (ollama.com)."""
        return "ollama.com" in self.base_url

    def set_few_shot_examples(self, texts, labels):
        """Selecciona ejemplos few-shot balanceados a partir del entrenamiento."""
        self.few_shot_examples = []
        unique_labels = sorted(set(labels))
        for label in unique_labels:
            indices = [i for i, l in enumerate(labels) if l == label]
            # Ejemplos cortos y diversos (los cortos suelen ser más limpios)
            selected = sorted(indices, key=lambda i: len(str(texts[i])))
            selected = selected[:self.num_few_shot]
            for idx in selected:
                self.few_shot_examples.append({"text": str(texts[idx]), "label": label})

    # ------------------------------------------------------------------
    # Construcción del prompt
    # ------------------------------------------------------------------
    def _build_chat_messages(self, text: str) -> list:
        """Construye la lista de mensajes chat con few-shot como turnos
        user/assistant separados. Este formato es el más universal: lo
        entienden bien tanto modelos chat-tuned como modelos instruidos
        con plantilla simple."""
        messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]

        for ex in self.few_shot_examples:
            messages.append({
                "role": "user",
                "content": self._USER_TEMPLATE.format(text=ex["text"]),
            })
            messages.append({
                "role": "assistant",
                "content": self._ASSISTANT_TEMPLATE.format(label=ex["label"]),
            })

        messages.append({
            "role": "user",
            "content": self._USER_TEMPLATE.format(text=text),
        })
        return messages

    def _build_flat_prompt(self, text: str) -> str:
        """Versión plana del prompt para APIs que no aceptan chat (HF Inference)."""
        parts = [self.SYSTEM_PROMPT, ""]
        for ex in self.few_shot_examples:
            parts.append(self._USER_TEMPLATE.format(text=ex["text"]))
            parts.append(self._ASSISTANT_TEMPLATE.format(label=ex["label"]))
            parts.append("")
        parts.append(self._USER_TEMPLATE.format(text=text))
        parts.append("<respuesta>")  # induce a completar
        return "\n".join(parts)

    # ------------------------------------------------------------------
    # Parser de respuesta robusto
    # ------------------------------------------------------------------
    _THINK_RE = None  # se inicializa lazy

    @classmethod
    def _strip_thinking(cls, text: str) -> str:
        """Elimina bloques de razonamiento (<think>, <thinking>, <reasoning>)
        emitidos por modelos como qwen3, deepseek-r1, gpt-oss. Sin esto,
        el parser podría encontrar la palabra 'ironia' dentro del razonamiento
        antes de la etiqueta final."""
        import re
        if cls._THINK_RE is None:
            cls._THINK_RE = re.compile(
                r"<(think|thinking|reasoning|reflection)>.*?</\1>",
                re.DOTALL | re.IGNORECASE,
            )
        # Caso 1: bloques bien cerrados -> los eliminamos
        cleaned = cls._THINK_RE.sub("", text)
        # Caso 2: solo aparece la apertura sin cierre (truncado por max_tokens):
        # nos quedamos con lo que haya tras el último marcador conocido de cierre.
        # Si no hay cierre, tomamos todo después del último <think> abierto;
        # si tampoco hay, devolvemos el texto original.
        lower = cleaned.lower()
        for opener in ("<think>", "<thinking>", "<reasoning>"):
            if opener in lower:
                # Si hay apertura pero no cierre, descartamos lo anterior al opener
                # y seguimos: probablemente el modelo terminó razonando dentro del
                # bloque (sin emitir <respuesta>). Mejor caer al fallback default.
                idx = lower.rfind(opener)
                cleaned = cleaned[idx + len(opener):]
                lower = cleaned.lower()
        return cleaned

    def _parse_response(self, response_text: str) -> str:
        """Extrae la etiqueta de la respuesta del LLM.

        Orden de búsqueda:
          1. Quita bloques <think>...</think>.
          2. Busca dentro de <respuesta>...</respuesta>.
          3. Fallback: busca tokens en la última línea no vacía.
          4. Defecto: 'no_ironia'.
        """
        import re
        if not response_text:
            return "no_ironia"

        cleaned = self._strip_thinking(response_text).strip()

        # 1) Etiqueta dentro de <respuesta>...</respuesta>
        m = re.search(
            r"<respuesta>\s*(.*?)\s*</respuesta>",
            cleaned,
            re.IGNORECASE | re.DOTALL,
        )
        if m:
            inner = m.group(1).strip().lower()
            # Normaliza tildes/espacios
            inner_norm = inner.replace("í", "i").replace(" ", "_")
            if "no_ironia" in inner_norm or inner_norm.startswith("no"):
                return "no_ironia"
            if "ironia" in inner_norm:
                return "ironia"

        # 2) Fallback: mira la última línea no vacía (modelos que ignoran tags)
        last_line = ""
        for line in reversed(cleaned.splitlines()):
            line = line.strip()
            if line:
                last_line = line
                break
        last_norm = last_line.lower().replace("í", "i")

        # IMPORTANTE: comprobar primero "no_ironia" / "no ironia" porque
        # "ironia" es subcadena de "no_ironia".
        if ("no_ironia" in last_norm
                or "no ironia" in last_norm
                or "no es ironia" in last_norm
                or last_norm.startswith("no")):
            return "no_ironia"
        if "ironia" in last_norm:
            return "ironia"

        # 3) Último intento: en todo el texto limpio
        full_norm = cleaned.lower().replace("í", "i")
        if "no_ironia" in full_norm or "no ironia" in full_norm:
            return "no_ironia"
        if "ironia" in full_norm:
            return "ironia"

        logger.warning(
            "Respuesta LLM no parseable (primeros 200c): %r. Default: no_ironia",
            response_text[:200],
        )
        return "no_ironia"

    # ------------------------------------------------------------------
    # Predicción
    # ------------------------------------------------------------------
    def predict_one(self, text: str) -> str:
        if self.provider == "ollama":
            return self._predict_ollama(text)
        elif self.provider == "huggingface":
            return self._predict_huggingface(text)
        else:
            raise ValueError(f"Unknown provider: {self.provider}")

    def _predict_ollama(self, text: str) -> str:
        """Llama al endpoint /api/chat de Ollama (local o Cloud).

        - Local:  POST {base_url}/api/chat              (sin headers)
        - Cloud:  POST https://ollama.com/api/chat      (Authorization: Bearer <key>)
        """
        import requests

        url = f"{self.base_url}/api/chat"
        messages = self._build_chat_messages(text)

        payload = {
            "model": self.model,
            "messages": messages,
            "stream": False,
            "options": {
                "temperature": self.temperature,
                "num_predict": self.max_tokens,
                # stop sequences: en cuanto cerramos </respuesta>, fin.
                "stop": ["</respuesta>"],
            },
        }

        headers = {"Content-Type": "application/json"}
        if self.api_key:
            # Ollama Cloud usa Bearer token. También se envía si está en local
            # con un proxy autenticado (e.g. Open WebUI).
            headers["Authorization"] = f"Bearer {self.api_key}"

        try:
            resp = requests.post(
                url, json=payload, headers=headers,
                timeout=self.request_timeout,
            )
            resp.raise_for_status()
            data = resp.json()
            content = data.get("message", {}).get("content", "")
            # Si activamos stop=</respuesta>, Ollama lo elimina del output:
            # reañadimos para que el parser lo encuentre.
            if "<respuesta>" in content and "</respuesta>" not in content:
                content = content + "</respuesta>"
            return self._parse_response(content)
        except requests.HTTPError as e:
            status = e.response.status_code if e.response is not None else "?"
            body = e.response.text[:300] if e.response is not None else ""
            logger.error(
                "Ollama HTTP %s en %s. ¿Modelo descargado? ¿API key válida? Body: %s",
                status, url, body,
            )
            return "no_ironia"
        except Exception as e:
            logger.error("Ollama error: %s", e)
            return "no_ironia"

    def _predict_huggingface(self, text: str) -> str:
        import requests
        url = f"https://api-inference.huggingface.co/models/{self.model}"
        headers = {}
        if self.hf_token:
            headers["Authorization"] = f"Bearer {self.hf_token}"

        prompt = self._build_flat_prompt(text)

        payload = {
            "inputs": prompt,
            "parameters": {
                "max_new_tokens": self.max_tokens,
                "temperature": max(self.temperature, 0.01),
                "return_full_text": False,
                "stop": ["</respuesta>"],
            },
        }
        try:
            resp = requests.post(
                url, json=payload, headers=headers,
                timeout=self.request_timeout,
            )
            resp.raise_for_status()
            data = resp.json()
            if isinstance(data, list) and len(data) > 0:
                content = data[0].get("generated_text", "")
            else:
                content = str(data)
            # Si el modelo solo devuelve la continuación tras "<respuesta>",
            # reconstruimos la etiqueta envuelta:
            if "<respuesta>" not in content:
                content = "<respuesta>" + content
            return self._parse_response(content)
        except Exception as e:
            logger.error("HuggingFace API error: %s", e)
            return "no_ironia"

    def predict_batch(self, texts, show_progress=True) -> list:
        predictions = []
        iterator = tqdm(texts, desc="LLM predictions") if show_progress else texts
        for text in iterator:
            pred = self.predict_one(text)
            predictions.append(pred)
        return predictions


# ============================================================================
# CLASSIFIER FACTORY
# ============================================================================

def get_classifier(name: str, grid_search: bool = True, cv_folds: int = 5):
    """Return a classifier (optionally wrapped in GridSearchCV)."""

    if name == "svm" and grid_search:
        param_grid = {
            "C": [0.1, 1, 10, 100],
            "gamma": ["scale", "auto", 0.01, 0.001],
            "kernel": ["rbf", "poly"],
        }
        return GridSearchCV(
            SVC(class_weight="balanced", probability=True),
            param_grid, cv=cv_folds, scoring="f1_macro",
            refit=True, verbose=0, n_jobs=-1,
        )
    elif name == "svm":
        return SVC(class_weight="balanced", probability=True, C=10, kernel="rbf")

    elif name == "logreg":
        if grid_search:
            param_grid = {"C": [0.01, 0.1, 1, 10, 100]}
            return GridSearchCV(
                LogisticRegression(class_weight="balanced", max_iter=2000, solver="lbfgs"),
                param_grid, cv=cv_folds, scoring="f1_macro",
                refit=True, verbose=0, n_jobs=-1,
            )
        return LogisticRegression(class_weight="balanced", max_iter=2000)

    elif name == "mlp":
        if grid_search:
            param_grid = {
                "hidden_layer_sizes": [(256, 128), (512, 256), (128, 64, 32)],
                "alpha": [1e-4, 1e-3, 1e-2],
            }
            return GridSearchCV(
                MLPClassifier(max_iter=500, early_stopping=True, validation_fraction=0.15),
                param_grid, cv=cv_folds, scoring="f1_macro",
                refit=True, verbose=0, n_jobs=-1,
            )
        return MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, early_stopping=True)

    elif name == "rf":
        if grid_search:
            param_grid = {
                "n_estimators": [100, 300, 500],
                "max_depth": [None, 10, 20],
            }
            return GridSearchCV(
                RandomForestClassifier(class_weight="balanced"),
                param_grid, cv=cv_folds, scoring="f1_macro",
                refit=True, verbose=0, n_jobs=-1,
            )
        return RandomForestClassifier(n_estimators=300, class_weight="balanced")

    elif name == "lgbm":
     from lightgbm import LGBMClassifier
     if grid_search:
        param_grid = {
            "n_estimators": [200, 500, 1000],
            "learning_rate": [0.01, 0.05, 0.1],
            "num_leaves": [31, 63, 127],
            "max_depth": [-1, 10, 20],
            "min_child_samples": [10, 20, 50],
            "reg_alpha": [0, 0.1, 1.0],
            "reg_lambda": [0, 0.1, 1.0],
        }
        return GridSearchCV(
            LGBMClassifier(
                class_weight="balanced",
                objective="multiclass",  # o "binary" según tu tarea
                random_state=42,
                n_jobs=-1,
                verbose=-1,
            ),
            param_grid, cv=cv_folds, scoring="f1_macro",
            refit=True, verbose=0, n_jobs=-1,
        )

     return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
     )

    else:
        raise ValueError(f"Unknown classifier: {name}")


def get_scaler(name: str):
    if name == "minmax":
        return MinMaxScaler()
    elif name == "standard":
        return StandardScaler()
    elif name == "none":
        return None
    else:
        raise ValueError(f"Unknown scaler: {name}")


# ============================================================================
# MAIN PIPELINE
# ============================================================================

class SpeechMATICSPipeline:
    """
    Main pipeline orchestrating text features, audio features,
    classifiers, and LLMs for both subtasks.
    """

    def __init__(self, config: dict):
        self.config = config
        self.label_map = {"ironia": 1, "no_ironia": 0}
        self.id2label = {0: "no_ironia", 1: "ironia"}
        self.label2id = {"ironia": 1, "no_ironia": 0}

    def load_data(self):
        logger.info("Loading datasets...")
        self.train_df = pd.read_csv(self.config["train_csv"])
        self.test_df = pd.read_csv(self.config["test_csv"])
        logger.info(f"Train: {len(self.train_df)} samples | Test: {len(self.test_df)} samples")

        # Show label distribution
        if "label" in self.train_df.columns:
            dist = self.train_df["label"].value_counts()
            logger.info(f"Label distribution:\n{dist}")

    # ---- TASK 1: TEXT-ONLY ----

    def run_task1(self) -> list:
        approach = self.config["task1_approach"]
        logger.info(f"=== TASK 1 (text-only) | Approach: {approach} ===")

        if approach == "tfidf":
            return self._task1_tfidf()
        elif approach == "embedding":
            return self._task1_embedding()
        elif approach == "llm":
            return self._task1_llm()
        elif approach == "ensemble":
            return self._task1_ensemble()
        else:
            raise ValueError(f"Unknown task1 approach: {approach}")

    def _task1_tfidf(self):
        extractor = TfidfFeatureExtractor(max_features=10_000, ngram_range=(1, 2))
        X_train = extractor.fit_transform(self.train_df["transcripcion"])
        X_test = extractor.transform(self.test_df["transcripcion"])
        return self._train_and_predict(X_train, self.train_df["label"], X_test)

    def _task1_embedding(self):
        extractor = EmbeddingFeatureExtractor(
            model_name=self.config["embedding_model"],
            device=self.config["embedding_device"],
            batch_size=self.config["embedding_batch_size"],
            normalize=self.config["embedding_normalize"],
        )
        print("Ejecutando embedings Task 1")
        X_train = extractor.fit_transform(self.train_df["transcripcion"].tolist())
        X_test = extractor.transform(self.test_df["transcripcion"].tolist())
        print(X_train)
        print(self.train_df["label"])
        return self._train_and_predict(X_train, self.train_df["label"], X_test)

    def _task1_llm(self):
        clf = LLMClassifier(
            provider=self.config["llm_provider"],
            model=self.config["llm_model"],
            base_url=self.config["ollama_base_url"],
            api_key=self.config.get("ollama_api_key"),
            hf_token=self.config.get("hf_api_token"),
            temperature=self.config["llm_temperature"],
            max_tokens=self.config["llm_max_tokens"],
            num_few_shot=self.config["llm_num_few_shot"],
            request_timeout=self.config.get("llm_request_timeout", 180),
        )
        # Set few-shot examples from training data
        clf.set_few_shot_examples(
            self.train_df["transcripcion"].tolist(),
            self.train_df["label"].tolist(),
        )
        predictions = clf.predict_batch(self.test_df["transcripcion"].tolist())
        return predictions

    def _task1_ensemble(self):
        """Combine multiple text approaches via voting."""
        methods = self.config.get("ensemble_methods", ["tfidf", "embedding"])
        all_preds = {}

        if "tfidf" in methods:
            logger.info("Ensemble component: TF-IDF")
            all_preds["tfidf"] = self._task1_tfidf()

        if "embedding" in methods:
            logger.info("Ensemble component: Embedding")
            all_preds["embedding"] = self._task1_embedding()

        if "llm" in methods:
            logger.info("Ensemble component: LLM")
            all_preds["llm"] = self._task1_llm()

        # Majority voting
        return self._majority_vote(all_preds)

    # ---- TASK 2: MULTIMODAL ----

    def run_task2(self, text_features_cache=None) -> list:
        approach = self.config["task2_approach"]
        logger.info(f"=== TASK 2 (multimodal) | Approach: {approach} ===")
        print("Ejecutando Tarea 2")
        if approach == "tfidf_mfcc":
            return self._task2_feature_based("tfidf")
        elif approach == "embedding_mfcc":
            return self._task2_feature_based("embedding")
        elif approach == "llm":
            return self._task2_llm()
        elif approach == "ensemble":
            return self._task2_ensemble()
        else:
            raise ValueError(f"Unknown task2 approach: {approach}")

    def _extract_audio_features(self):
        """Extract audio features for train and test sets."""
        audio_ext = AudioFeatureExtractor(
            features=self.config["audio_features"],
            n_mfcc=self.config["n_mfcc"],
            sr=self.config["audio_sr"],
        )
        X_audio_train, y_train = audio_ext.extract_batch(
            self.train_df["id"],
            self.config["train_audio_dir"],
            self.train_df["label"],
            self.label2id,
        )
        X_audio_test, _ = audio_ext.extract_batch(
            self.test_df["id"],
            self.config["test_audio_dir"],
        )
        return X_audio_train, X_audio_test, y_train

    def _task2_feature_based(self, text_method="embedding"):
        """Combine text features + audio features, then classify."""
        # Text features
        if text_method == "embedding":
            text_ext = EmbeddingFeatureExtractor(
                model_name=self.config["embedding_model"],
                device=self.config["embedding_device"],
                batch_size=self.config["embedding_batch_size"],
                normalize=self.config["embedding_normalize"],
            )
        else:
            text_ext = TfidfFeatureExtractor(max_features=10_000, ngram_range=(1, 2))

        X_text_train = text_ext.fit_transform(self.train_df["transcripcion"].tolist())
        X_text_test = text_ext.transform(self.test_df["transcripcion"].tolist())

        # Audio features
        X_audio_train, X_audio_test, _ = self._extract_audio_features()

        # Concatenate
        X_train = np.concatenate([X_text_train, X_audio_train], axis=1)
        X_test = np.concatenate([X_text_test, X_audio_test], axis=1)

        return self._train_and_predict(X_train, self.train_df["label"], X_test)

    def _task2_llm(self):
        """Use LLM for multimodal task (text-only, since LLMs can't process audio directly)."""
        logger.warning(
            "LLM approach for Task 2 uses text only. "
            "For true multimodal, use 'embedding_mfcc' or 'ensemble'."
        )
        return self._task1_llm()

    def _task2_ensemble(self):
        """Ensemble of multimodal approaches."""
        all_preds = {}

        logger.info("Ensemble component: TF-IDF + MFCC")
        all_preds["tfidf_mfcc"] = self._task2_feature_based("tfidf")

        logger.info("Ensemble component: Embedding + MFCC")
        all_preds["embedding_mfcc"] = self._task2_feature_based("embedding")

        return self._majority_vote(all_preds)

    # ---- SHARED UTILITIES ----

    def _train_and_predict(self, X_train, y_train_labels, X_test) -> list:
        """Scale features, train classifier, predict, return label strings."""
        # Encode labels
        y_train = np.array([self.label2id[l] for l in y_train_labels])

        # Scale
        scaler = get_scaler(self.config["scaler"])
        if scaler:
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        # Classifier
        clf = get_classifier(
            self.config["classifier"],
            grid_search=self.config["grid_search"],
            cv_folds=self.config["cv_folds"],
        )

        logger.info(f"Training classifier: {self.config['classifier']} ...")
        clf.fit(X_train, y_train)

        # Report best params if grid search
        if hasattr(clf, "best_params_"):
            logger.info(f"Best params: {clf.best_params_}")
            logger.info(f"Best CV score: {clf.best_score_:.4f}")

        # Cross-validation score on train
        if hasattr(clf, "best_estimator_"):
            best = clf.best_estimator_
        else:
            best = clf
        cv_scores = cross_val_score(best, X_train, y_train, cv=5, scoring="f1_macro")
        logger.info(f"CV F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

        # Predict
        y_pred = clf.predict(X_test)
        predictions = [self.id2label[p] for p in y_pred]
        return predictions

    def _majority_vote(self, all_preds: dict) -> list:
        """Simple majority voting across multiple prediction lists."""
        method_names = list(all_preds.keys())
        n_samples = len(list(all_preds.values())[0])
        final = []

        for i in range(n_samples):
            votes = [all_preds[m][i] for m in method_names]
            ironic_count = sum(1 for v in votes if v == "ironia")
            final.append("ironia" if ironic_count > len(votes) / 2 else "no_ironia")

        return final

    # ---- SUBMISSION ----

    def create_submission(self, task1_preds, task2_preds):
        """Create CodaBench submission CSV."""
        output_df = pd.DataFrame()
        output_df["id"] = self.test_df["id"]
        output_df["label_t1"] = task1_preds
        output_df["label_t2"] = task2_preds

        out_path = self.config["output_file"]
        output_df.to_csv(out_path, index=False, header=False)
        logger.info(f"Submission saved to: {out_path}")

        # Also create zip for CodaBench
        import zipfile
        zip_path = out_path.replace(".csv", ".zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(out_path, os.path.basename(out_path))
        logger.info(f"Submission zip: {zip_path}")

        # Quick stats
        logger.info(f"Task 1 distribution: {pd.Series(task1_preds).value_counts().to_dict()}")
        logger.info(f"Task 2 distribution: {pd.Series(task2_preds).value_counts().to_dict()}")

    def run(self):
        """Execute the full pipeline."""
        self.load_data()
        task1_preds = self.run_task1()
        task2_preds = self.run_task2()
        self.create_submission(task1_preds, task2_preds)
        logger.info("Pipeline complete!")

def main():
   config = load_config()
   pipeline = SpeechMATICSPipeline(config)
   pipeline.run()

if __name__ == "__main__":
    main()

Ejecutando embedings Task 1


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/160 [00:00<?, ?it/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

[[ 0.03482   0.01112  -0.005833 ... -0.01979  -0.0482    0.03857 ]
 [ 0.01811  -0.001052  0.00815  ... -0.0187   -0.02841   0.02252 ]
 [ 0.01543  -0.006584 -0.0394   ... -0.001697 -0.04474   0.02266 ]
 ...
 [ 0.01662  -0.02255  -0.0216   ... -0.00576  -0.04123   0.02899 ]
 [ 0.01671   0.01814  -0.003191 ...  0.008934 -0.0335    0.01968 ]
 [ 0.00498   0.002583 -0.02942  ... -0.00712  -0.02998   0.01569 ]]
0          ironia
1          ironia
2          ironia
3       no_ironia
4          ironia
          ...    
5095       ironia
5096    no_ironia
5097       ironia
5098       ironia
5099    no_ironia
Name: label, Length: 5100, dtype: object


LLM predictions:  82%|████████▏ | 739/903 [35:09<05:50,  2.13s/it]

In [ ]:

# ============================================================================
# CLI
# ============================================================================

def parse_args():
    parser = argparse.ArgumentParser(
        description="SpeechMATICS Advanced Pipeline for Irony Detection"
    )
    parser.add_argument("--config", type=str, default=None,
                        help="Path to YAML/JSON config file")
    parser.add_argument("--task1-approach", type=str,
                        choices=["tfidf", "embedding", "llm", "ensemble"],
                        help="Approach for Task 1")
    parser.add_argument("--task2-approach", type=str,
                        choices=["tfidf_mfcc", "embedding_mfcc", "llm", "ensemble"],
                        help="Approach for Task 2")
    parser.add_argument("--embedding-model", type=str,
                        help="HuggingFace model for embeddings")
    parser.add_argument("--llm-provider", type=str, choices=["ollama", "huggingface"],
                        help="LLM provider")
    parser.add_argument("--llm-model", type=str,
                        help="LLM model name")
    parser.add_argument("--ollama-base-url", type=str,
                        help="Ollama base URL. Usa http://localhost:11434 para "
                             "local o https://ollama.com para Ollama Cloud.")
    parser.add_argument("--ollama-api-key", type=str,
                        help="API key para Ollama Cloud. Alternativa: env OLLAMA_API_KEY.")
    parser.add_argument("--classifier", type=str,
                        choices=["svm", "logreg", "mlp", "rf"],
                        help="Classifier for feature-based approaches")
    parser.add_argument("--device", type=str, choices=["cpu", "cuda", "mps"],
                        help="Device for embeddings")
    parser.add_argument("--output", type=str,
                        help="Output submission file path")
    parser.add_argument("--no-grid-search", action="store_true",
                        help="Disable grid search (faster)")
    return parser.parse_args()


def main():
    args = parse_args()
    config = load_config(args.config)

    # CLI overrides
    if args.task1_approach:
        config["task1_approach"] = args.task1_approach
    if args.task2_approach:
        config["task2_approach"] = args.task2_approach
    if args.embedding_model:
        config["embedding_model"] = args.embedding_model
    if args.llm_provider:
        config["llm_provider"] = args.llm_provider
    if args.llm_model:
        config["llm_model"] = args.llm_model
    if args.ollama_base_url:
        config["ollama_base_url"] = args.ollama_base_url
    if args.ollama_api_key:
        config["ollama_api_key"] = args.ollama_api_key
    if args.classifier:
        config["classifier"] = args.classifier
    if args.device:
        config["embedding_device"] = args.device
    if args.output:
        config["output_file"] = args.output
    if args.no_grid_search:
        config["grid_search"] = False

    logger.info("Configuration:")
    for k, v in sorted(config.items()):
        if "token" not in k.lower() and "api_key" not in k.lower():
            logger.info(f"  {k}: {v}")

    pipeline = SpeechMATICSPipeline(config)
    pipeline.run()
